# NFG explained

#### 1. Initialize environment & Get joint information

In [ ]:
from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
# trajectory tools 
from ri_motion_v5_package.traj_optimization.traj_utils import *
from ri_motion_v5_package.traj_optimization.traj_shaper import *
# gp tools 
from ri_motion_v5_package.gaussian_process.gp_utils import *
from ri_motion_v5_package.gaussian_process.kernels import rbf_kernel

from package.panda_env import * 

from PyQt5.QtWidgets import QApplication
app = QApplication(sys.argv)

from package.nfg import *

In [ ]:
xml_path = merge_mjcfs(
    included_mjcf_files=[
        "../../../asset/floor/floor_white_gray.xml",
        "./asset/doosan/doosan_tesollo_dg5g_with_base.xml",
        "./asset/franka_left_inspire/franka_left_inspire_with_base_site_added.xml",
        "./asset/object/obj_table.xml",
        # "./asset/b10_nfg_explained.ipynbmw_object/bmw_washer_nut.xml",
        "./asset/bmw_object/bmw_washer_nut_holder.xml",
        "./asset/bmw_object/bmw_housing.xml",
        "./asset/bmw_object/bmw_part.xml",
        "./asset/bmw_object/bmw_bosch_driver.xml",
        "./asset/bmw_object/bmw_jig.xml",
    ],
    output_xml_path = "xml/scene_rlwrld_bmw.xml",
)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=True)

In [ ]:
def set_robot_q_obj_pR_bmw_env(env):
    """ 
    Later changed to env.reset()
    """
    env.reset()
    env.set_p(type="body",name='obj_table',p=[0.825,0.0,0.0])
    env.set_T(type="body",name='robot_frk:mobile_base',T=pr2t(p=[1.486,0,0],R=rpy2r([0,0,180*D2R])))
    env.forward(q=[1.57,1.57],joint_names=['robot_doosan:joint_3','robot_doosan:joint_5'])
    env.forward(q=[-1.57,1.57],joint_names=['robot_frk:joint4','robot_frk:joint6'])
    # env.set_p(type="base",name='bmw_washer_nut',p=[0.93,0.2,0.8017])
    env.set_p(type="base",name='bmw_washer_nut_holder',p=[1,0.2,0.799])
    env.set_p(type="base",name='bmw_housing',p=[0.7,-0.5,0.804])
    env.set_T(type="base",name='bmw_part',T=pr2t(p=[0.9,0.08,0.803],R=rpy2r([1.57,0,1.57])))
    env.set_p(type="base",name='bmw_bosch_driver',p=[0.65,0,0.8])
    env.set_T(type="base",name='bmw_jig',T=pr2t(p=[0.85,-0.2,0.8],R=rpy2r([1.57,0,0])))
    # Trim joints
    q = np.clip(env.get_qpos(env.rev_joint_names),env.rev_joint_mins,env.rev_joint_maxs)
    env.forward(q=q,joint_names=env.rev_joint_names)

def get_R_franka_lpalm(env):
    p_palm        = env.get_p('lpalm','site')
    p_palm_top    = env.get_p('lpalm_top','site')
    p_palm_palmar = env.get_p('lpalm_palmar','site')
    p_palm_front  = env.get_p('lpalm_front','site')
    ux,uy,uz      = np_uv(p_palm_front-p_palm),np_uv(p_palm-p_palm_palmar),np_uv(p_palm_top-p_palm)
    R_franka_lpalm = np.column_stack((ux,uy,uz)).reshape((3,3))
    return R_franka_lpalm

#### 2. Get joint names & set EE pR

In [ ]:
# get joint names
franka_joints = [
    'robot_frk:joint1','robot_frk:joint2','robot_frk:joint3','robot_frk:joint4',
    'robot_frk:joint5','robot_frk:joint6','robot_frk:joint7',
]
franka_body_idxs = env.get_body_idxs_from_root_body_name('fr3_link1')

# get q 
q_franka_initial = [0.0, 0.0, 0.0, -1.57, 0.0, 1.57, 0.0]
q_franka_pick = [-0.48,  0.18, -0.34, -2.63,  2.1,   2.08, -0.46]
q_franka_place = [-1.35, -1.5,   1.91, -1.92,  2.76,  3.02,  1.14]

#### 3. Main Loop

In [ ]:
""" TRAJECTORY 1"""
n_traj = 20
n_iter = 5
topk                = 5
trim_scale          = True
variance            = 0.3
length_scale_coef   = 0.5
contact_gain        = 10.0
motion_gain         = 5.0
q_min               = env.get_joint_mins(franka_joints)
q_max               = env.get_joint_maxs(franka_joints)

In [ ]:
""" TRAJECTORY 1"""
set_robot_q_obj_pR_bmw_env(env)
# set anchors and get smoothed trajectory 
anchors = np.vstack((q_franka_initial,q_franka_pick))
L, times, traj_smt = get_smooth_traj_from_anchors(
    anchors = anchors,
    freq = 100,
    time = 5.0,
)
traj_update = traj_smt.copy()
# Get covariance matrix
K_chol, t_out = get_covariance_matrix_time_hilbert_space(
    times = times,
    L = L,
    length_scale_coef = length_scale_coef,
    variance = variance,
    kernel = rbf_kernel,
)
# loop: calculate costs, update traj, repeat
for iter_idx in range(n_iter):
    epsilon, traj_multidim = sample_epsilon_squash_trajs_multidim(
        traj=traj_update,
        K_chol=K_chol,
        n_traj=n_traj,
        dim=len(franka_joints),
        q_min=q_min,
        q_max=q_max,
        use_uniform=True,
        seed=iter_idx,
        squash_margin=0.1,
    )
    # calculate scores
    motion_scores = calculate_motion_score(traj_multidim)
    contact_scores = np.zeros(n_traj)
    for traj_idx in range(n_traj):
        traj = traj_multidim[:,traj_idx,:]
        score_traj_single = np.zeros(L)
        for time_idx in range(L):
            env.forward(q=traj[time_idx], joint_names=franka_joints)
            contact_info = env.get_contact_info()
            body_idxs = env.get_body_idxs_from_root_body_name('fr3_link1')
            contact_info = filter_contact_info_with_body_idxs(contact_info, body_idxs=body_idxs)
            score_current_t = calculate_contact_score(contact_info)
            score_traj_single[time_idx] = score_current_t
        contact_scores[traj_idx] = score_traj_single.min()
    # if solved, terminalize
    if np.any(contact_scores  == 0.0):
        print_green(f"\n Contact-free trajectory found at iteration {iter_idx}, terminalizing optimization.")
        solved_idxs = np.where(contact_scores == 0.0)[0]
        best_solved_idx = solved_idxs[np.argmin(motion_scores[solved_idxs])]
        traj_update = traj_multidim[:,best_solved_idx,:]
        break
    # calculate gradient and update trajectory
    scores = contact_gain * contact_scores - motion_gain * motion_scores
    traj_update = update_traj_with_calculated_gradient(
        traj=traj_update,
        scores = scores,
        epsilon=epsilon,
        step_size=0.5,
    )
    print_yellow(f"Iteration {iter_idx+1}/{n_iter}, score_sum: {scores.sum():.4f}, score_max: {scores.max():.4f}, score_min: {scores.min():.4f}", end="")
traj_best_1 = traj_update

In [ ]:
""" TRAJECTORY 2 """
n_traj = 20
n_iter = 5
topk                = 5
trim_scale          = True
variance            = 1.0
length_scale_coef   = 0.5
contact_gain        = 10.0
motion_gain         = 5.0
q_min               = env.get_joint_mins(franka_joints)
q_max               = env.get_joint_maxs(franka_joints)

In [ ]:
""" TRAJECTORY 2 """
set_robot_q_obj_pR_bmw_env(env)
# set anchors and get smoothed trajectory 
anchors = np.vstack((q_franka_pick, q_franka_place))
L, times, traj_smt = get_smooth_traj_from_anchors(
    anchors = anchors,
    freq = 100,
    time = 5.0,
)
traj_update = traj_smt.copy()
# Get covariance matrix
K_chol, t_out = get_covariance_matrix_time_hilbert_space(
    times = times,
    L = L,
    length_scale_coef = length_scale_coef,
    variance = variance,
    kernel = rbf_kernel,
)
# loop: calculate costs, update traj, repeat
for iter_idx in range(n_iter):
    epsilon, traj_multidim = sample_epsilon_squash_trajs_multidim(
        traj=traj_update,
        K_chol=K_chol,
        n_traj=n_traj,
        dim=len(franka_joints),
        q_min=q_min,
        q_max=q_max,
        use_uniform=True,
        seed=iter_idx,
        squash_margin=0.1,
    )
    # calculate scores
    motion_scores = calculate_motion_score(traj_multidim)
    contact_scores = np.zeros(n_traj)
    for traj_idx in range(n_traj):
        traj = traj_multidim[:,traj_idx,:]
        score_traj_single = np.zeros(L)
        for time_idx in range(L):
            env.forward(q=traj[time_idx], joint_names=franka_joints)
            contact_info = env.get_contact_info()
            body_idxs = env.get_body_idxs_from_root_body_name('fr3_link1')
            contact_info = filter_contact_info_with_body_idxs(contact_info, body_idxs=body_idxs)
            score_current_t = calculate_contact_score(contact_info)
            score_traj_single[time_idx] = score_current_t
        contact_scores[traj_idx] = score_traj_single.min()
    # if solved, terminalize
    if np.any(contact_scores  == 0.0):
        print_green(f"\n Contact-free trajectory found at iteration {iter_idx}, terminalizing optimization.")
        solved_idxs = np.where(contact_scores == 0.0)[0]
        best_solved_idx = solved_idxs[np.argmin(motion_scores[solved_idxs])]
        traj_update = traj_multidim[:,best_solved_idx,:]
        break
    # calculate gradient and update trajectory
    scores = contact_gain * contact_scores - motion_gain * motion_scores
    traj_update = update_traj_with_calculated_gradient(
        traj=traj_update,
        scores = scores,
        epsilon=epsilon,
        step_size=0.5,
    )
    print_yellow(f"Iteration {iter_idx+1}/{n_iter}, score_sum: {scores.sum():.4f}, score_max: {scores.max():.4f}, score_min: {scores.min():.4f}", end="")
traj_best_2 = traj_update

#### 5. Visualize trajectory

In [ ]:
traj = np.concatenate((traj_best_1, traj_best_2), axis=0)
set_robot_q_obj_pR_bmw_env(env)
env.init_viewer(title='Example',width=0.6,height=1.0,fontscale=200)
env.viewer.set_transparency(transparent=False)
env.viewer.set_cam_info(azimuth=-146,distance=3.3,elevation=-34,lookat=[-0.05,-0.01,1.02])

idx = 0
running_flag = False
total_contact = 0
robot_contact = 0
while env.is_viewer_alive():
    # Update
    if env.is_key_pressed_once(glfw.KEY_ESCAPE): 
        final_rgb_img,_ = env.grab_current_rgbd_imgs()
        break
    if env.is_key_pressed_once(glfw.KEY_SPACE):
        running_flag = not running_flag
    if env.is_key_pressed_once(glfw.KEY_R):
        idx = 0
        set_robot_q_obj_pR_bmw_env(env)
        running_flag = False
    if running_flag:
        if idx < len(traj):
            q_curr = traj[idx]
            env.forward(q=q_curr, joint_names=franka_joints)
            idx += 1
    contact_info = env.get_contact_info()
    n_contact_curr = contact_info['n_contact']
    total_contact += n_contact_curr
    body_idxs = env.get_body_idxs_from_root_body_name('fr3_link1')
    contact_info = filter_contact_info_with_body_idxs(contact_info, body_idxs=body_idxs)
    n_contact_curr = contact_info['n_contact']
    robot_contact += n_contact_curr
    env.viewer_text_overlay(f'# of robot contact: {robot_contact}')
    env.plot_contact_info(contact_info)
    env.plot_global_coordinate_axes()
    env.render()

imshow(final_rgb_img,fig_w=6,title='Final scene',title_fs=8)
env.close_viewer()